# EchoSeg-RT on a Kaggle GPU

In the right-hand Settings panel set **Accelerator: GPU T4 x2** and **Internet: On**, then run the cells in order. About 60 minutes end to end.

In [ ]:
REPO = 'https://github.com/YOUR_USERNAME/echoseg-rt.git'  # <- your public repository
%cd /kaggle/working
!rm -rf echoseg-rt
!git clone $REPO echoseg-rt
%cd echoseg-rt
!pip install -q segmentation-models-pytorch nibabel
!nvidia-smi --query-gpu=name --format=csv

In [ ]:
!pytest -q

Download and preprocess CAMUS. Use `--n 150` if you are short on time.

In [ ]:
!python scripts/download_camus.py --out /kaggle/tmp/raw
!python -m echoseg.data preprocess --raw /kaggle/tmp/raw --out data/processed

Train both models at the same time, one per T4. The temporal-context model runs in the background on GPU 1 and logs to `ctx2.log`; the single-frame model prints live from GPU 0.

In [ ]:
import os, subprocess
ctx2 = subprocess.Popen(
    'python -u -m echoseg.train --set data.context=2 run_name=unet_r34_ctx2 > ctx2.log 2>&1',
    shell=True, env={**os.environ, 'CUDA_VISIBLE_DEVICES': '1'})
!CUDA_VISIBLE_DEVICES=0 python -u -m echoseg.train
print('waiting for the temporal-context run, exit code', ctx2.wait())
!tail -n 12 ctx2.log

Evaluate both on the held-out test patients.

In [ ]:
!python -m echoseg.evaluate --checkpoint runs/unet_r34_ctx0/best.pt
!python -m echoseg.evaluate --checkpoint runs/unet_r34_ctx2/best.pt

In [ ]:
!python -m echoseg.benchmark --checkpoint runs/unet_r34_ctx0/best.pt

In [ ]:
!python -m echoseg.demo --checkpoint runs/unet_r34_ctx0/best.pt --out assets/demo.gif
from IPython.display import Image
Image('assets/demo.gif')

Bundle the reports, benchmark and GIF. Click the link below to download, then copy the numbers into README.md and MODEL_CARD.md.

In [ ]:
!zip -q /kaggle/working/results.zip assets/demo.gif runs/*/report_test.* runs/*/benchmark.md runs/*/log.csv runs/*/split.json
from IPython.display import FileLink
%cd /kaggle/working
FileLink('results.zip')